# Experiment 13 — GLCM local neighborhood 3×3×3 vs 11×11×11 + XGBoost

Purpose: bridge Rao (3×3×3 local radiomics) and the thesis (`kernelRadius=5`, 11×11×11).

This experiment keeps:
- same BraTS 2021 577-patient MGMT cohort
- FLAIR only
- whole-tumor segmentation
- same 10 stratified patient folds
- same 24 GLCM feature definitions
- same patient-level representation: `%A` activation proportion
- same XGBoost model

Only the local GLCM neighborhood changes:
- K1 = 3×3×3
- K5 = 11×11×11


## 1. Install

In [ ]:
# !pip install numpy pandas scipy SimpleITK pyradiomics scikit-learn xgboost


## 2. Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage as ndi
from radiomics import featureextractor

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier


## 3. Configuration

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 10

MANIFEST_PATH = Path("/path/to/brats2021_577_manifest.csv")  # patient_id,flair_path,seg_path,mgmt
OUTDIR = Path("outputs_exp13_glcm_k1_vs_k5")
OUTDIR.mkdir(parents=True, exist_ok=True)

FOLD_ASSIGNMENT_CSV = OUTDIR / "fold_assignment_10fold.csv"

KERNELS = {
    "K1_3x3x3": 1,
    "K5_11x11x11": 5,
}

GLCM_FEATURES = [
    "Autocorrelation","ClusterProminence","ClusterShade","ClusterTendency",
    "Contrast","Correlation","DifferenceAverage","DifferenceEntropy",
    "DifferenceVariance","Id","Idm","Idmn","Idn","Imc1","Imc2",
    "InverseVariance","JointAverage","JointEnergy","JointEntropy","MCC",
    "MaximumProbability","SumAverage","SumEntropy","SumSquares",
]

PYRAD_SETTINGS = {
    "binWidth": 25,
    "normalize": False,
    "resampledPixelSpacing": None,
    "force2D": False,
    "distances": [1],
    "maskedKernel": False,
}


## 4. Load cohort

In [ ]:
def load_manifest(path):
    df = pd.read_csv(path)
    req = {"patient_id","flair_path","seg_path","mgmt"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    df["patient_id"] = df["patient_id"].astype(str)
    df["mgmt"] = df["mgmt"].astype(int)
    return df

manifest = load_manifest(MANIFEST_PATH)
print("Patients:", len(manifest))
print(manifest["mgmt"].value_counts().sort_index())
display(manifest.head())


## 5. Prepare 33×33×33 whole-tumor crop

The same fixed-size crop as the Rao reproduction is used so that K1 and K5 are compared on the same spatial grid.


In [ ]:
def bbox_from_mask(mask):
    coords = np.argwhere(mask)
    if not len(coords):
        raise ValueError("Empty tumor mask")
    z0,y0,x0 = coords.min(axis=0)
    z1,y1,x1 = coords.max(axis=0) + 1
    return (x0,y0,z0), (x1,y1,z1)

def cube_bounds(mask):
    (x0,y0,z0),(x1,y1,z1)=bbox_from_mask(mask)
    side=max(x1-x0,y1-y0,z1-z0)
    cx=(x0+x1-1)/2; cy=(y0+y1-1)/2; cz=(z0+z1-1)/2
    starts=[int(round(cx-(side-1)/2)),int(round(cy-(side-1)/2)),int(round(cz-(side-1)/2))]
    dims=[mask.shape[2],mask.shape[1],mask.shape[0]]
    for d in range(3):
        starts[d]=max(0,starts[d])
        if starts[d]+side>dims[d]:
            starts[d]=max(0,dims[d]-side)
    size=[min(side,dims[d]-starts[d]) for d in range(3)]
    return tuple(starts),tuple(size)

def crop_sitk(image,start,size):
    return sitk.RegionOfInterest(image,size=[int(v) for v in size],index=[int(v) for v in start])

def resample33(image,is_mask=False):
    old_size=np.array(image.GetSize(),float)
    old_spacing=np.array(image.GetSpacing(),float)
    new_size=np.array([33,33,33],int)
    new_spacing=old_spacing*old_size/new_size

    r=sitk.ResampleImageFilter()
    r.SetSize([33,33,33])
    r.SetOutputSpacing([float(v) for v in new_spacing])
    r.SetOutputOrigin(image.GetOrigin())
    r.SetOutputDirection(image.GetDirection())
    r.SetTransform(sitk.Transform())
    r.SetDefaultPixelValue(0)
    r.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    return r.Execute(sitk.Cast(image, sitk.sitkUInt8 if is_mask else sitk.sitkFloat32))

def prepare33(flair_path,seg_path):
    flair=sitk.ReadImage(str(flair_path))
    seg=sitk.ReadImage(str(seg_path))
    seg_arr=sitk.GetArrayFromImage(seg)
    tumor=seg_arr!=0
    start,size=cube_bounds(tumor)
    fc=crop_sitk(flair,start,size)
    sc=crop_sitk(seg,start,size)
    mask_arr=(sitk.GetArrayFromImage(sc)!=0).astype(np.uint8)
    mask=sitk.GetImageFromArray(mask_arr); mask.CopyInformation(sc)
    return resample33(fc,False),resample33(mask,True)


## 6. Global GLCM medians for activation thresholds

In [ ]:
def build_global_glcm_extractor():
    ex=featureextractor.RadiomicsFeatureExtractor(**PYRAD_SETTINGS)
    ex.disableAllFeatures()
    ex.enableFeaturesByName(glcm=GLCM_FEATURES)
    ex.disableAllImageTypes()
    ex.enableImageTypeByName("Original")
    return ex

def extract_global_glcm(manifest):
    out=OUTDIR/"global_glcm_24.csv"
    if out.exists():
        return pd.read_csv(out)

    ex=build_global_glcm_extractor()
    rows=[]
    for i,row in manifest.iterrows():
        print(f"[Global {i+1}/{len(manifest)}] {row.patient_id}")
        flair=sitk.ReadImage(row.flair_path)
        seg=sitk.ReadImage(row.seg_path)
        arr=(sitk.GetArrayFromImage(seg)!=0).astype(np.uint8)
        mask=sitk.GetImageFromArray(arr); mask.CopyInformation(seg)
        res=ex.execute(flair,mask)
        rec={"patient_id":row.patient_id,"mgmt":int(row.mgmt)}
        for k,v in res.items():
            if k.startswith("original_glcm_"):
                rec[k.replace("original_glcm_","")]=float(v)
        rows.append(rec)

    df=pd.DataFrame(rows)
    df.to_csv(out,index=False)
    return df

global_glcm=extract_global_glcm(manifest)
global_medians={c:float(global_glcm[c].median()) for c in GLCM_FEATURES if c in global_glcm}
display(global_glcm.head())


## 7. Local GLCM maps and patient-level `%A`

In [ ]:
def eligible_mask(tumor_img,radius):
    arr=sitk.GetArrayFromImage(tumor_img).astype(np.uint8)
    k=2*radius+1
    count=ndi.convolve(arr,np.ones((k,k,k),dtype=np.uint16),mode="constant",cval=0)
    eligible=(count>=1).astype(np.uint8)

    # Require full neighborhood inside the 33^3 volume
    if radius>0:
        eligible[:radius,:,:]=0; eligible[-radius:,:,:]=0
        eligible[:,:radius,:]=0; eligible[:,-radius:,:]=0
        eligible[:,:,:radius]=0; eligible[:,:,-radius:]=0

    out=sitk.GetImageFromArray(eligible)
    out.CopyInformation(tumor_img)
    return out

def build_local_glcm_extractor(radius):
    settings=dict(PYRAD_SETTINGS)
    settings["kernelRadius"]=radius
    ex=featureextractor.RadiomicsFeatureExtractor(**settings)
    ex.disableAllFeatures()
    ex.enableFeaturesByName(glcm=GLCM_FEATURES)
    ex.disableAllImageTypes()
    ex.enableImageTypeByName("Original")
    return ex

def map_values_at_mask(fmap,mask):
    arr=sitk.GetArrayFromImage(mask).astype(bool)
    vals=[]
    for z,y,x in np.argwhere(arr):
        pt=mask.TransformIndexToPhysicalPoint((int(x),int(y),int(z)))
        try:
            idx=fmap.TransformPhysicalPointToIndex(pt)
        except RuntimeError:
            continue
        size=fmap.GetSize()
        if any(idx[d]<0 or idx[d]>=size[d] for d in range(3)):
            continue
        vals.append(float(fmap.GetPixel(idx)))
    return np.asarray(vals,float)

def extract_activation_for_kernel(manifest,label,radius):
    out=OUTDIR/f"local_glcm_activation_{label}.csv"
    if out.exists():
        return pd.read_csv(out)

    ex=build_local_glcm_extractor(radius)
    rows=[]

    for i,row in manifest.iterrows():
        print(f"[{label} {i+1}/{len(manifest)}] {row.patient_id}")
        flair33,tumor33=prepare33(row.flair_path,row.seg_path)
        centers=eligible_mask(tumor33,radius)

        res=ex.execute(flair33,centers,label=1,voxelBased=True)

        rec={"patient_id":row.patient_id,"mgmt":int(row.mgmt)}

        for k,v in res.items():
            if not (k.startswith("original_glcm_") and isinstance(v,sitk.Image)):
                continue
            feat=k.replace("original_glcm_","")
            vals=map_values_at_mask(v,centers)
            vals=vals[np.isfinite(vals)]
            if len(vals)==0 or feat not in global_medians:
                rec[feat]=np.nan
            else:
                rec[feat]=float(np.mean(vals>=global_medians[feat]))

        rows.append(rec)

    df=pd.DataFrame(rows)
    df.to_csv(out,index=False)
    return df

activation={}
for label,radius in KERNELS.items():
    activation[label]=extract_activation_for_kernel(manifest,label,radius)
    print(label, activation[label].shape)


## 8. Create/reuse identical folds

In [ ]:
def make_folds(manifest,path):
    if path.exists():
        return pd.read_csv(path)
    skf=StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_STATE)
    rows=[]
    y=manifest["mgmt"].to_numpy()
    for fold,(_,va) in enumerate(skf.split(np.zeros(len(y)),y),1):
        for idx in va:
            rows.append({"patient_id":manifest.iloc[idx].patient_id,"fold":fold})
    out=pd.DataFrame(rows)
    out.to_csv(path,index=False)
    return out

folds=make_folds(manifest,FOLD_ASSIGNMENT_CSV)


## 9. XGBoost evaluation

In [ ]:
def build_xgb():
    return XGBClassifier(
        n_estimators=500,max_depth=6,learning_rate=0.05,
        min_child_weight=1,gamma=0,subsample=0.8,colsample_bytree=0.8,
        reg_alpha=0,reg_lambda=1,objective="binary:logistic",
        eval_metric="aucpr",tree_method="hist",
        random_state=RANDOM_STATE,n_jobs=-1
    )

def specificity(y,p):
    tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    return tn/(tn+fp) if tn+fp else np.nan

rows=[]

for label,df in activation.items():
    feature_cols=[c for c in GLCM_FEATURES if c in df.columns]
    work=df.dropna(subset=feature_cols).merge(folds,on="patient_id",validate="one_to_one")

    for fold in range(1,N_SPLITS+1):
        tr=work[work.fold!=fold]
        va=work[work.fold==fold]

        Xtr=tr[feature_cols].to_numpy(np.float32)
        ytr=tr.mgmt.to_numpy(int)
        Xva=va[feature_cols].to_numpy(np.float32)
        yva=va.mgmt.to_numpy(int)

        # Mirrors thesis weighting logic.
        classes=np.array([0,1])
        cwv=compute_class_weight("balanced",classes=classes,y=ytr)
        cw={0:float(cwv[0]),1:float(cwv[1])}
        sw=np.array([cw[int(v)] for v in ytr],dtype=np.float32)

        m=build_xgb()
        m.fit(Xtr,ytr,sample_weight=sw)

        prob=m.predict_proba(Xva)[:,1]
        pred=(prob>=0.5).astype(int)

        rows.append({
            "kernel":label,"fold":fold,
            "accuracy":accuracy_score(yva,pred),
            "precision":precision_score(yva,pred,zero_division=0),
            "recall":recall_score(yva,pred,zero_division=0),
            "specificity":specificity(yva,pred),
            "f1":f1_score(yva,pred,zero_division=0),
            "roc_auc":roc_auc_score(yva,prob),
            "pr_auc":average_precision_score(yva,prob),
            "balanced_accuracy":balanced_accuracy_score(yva,pred),
            "mcc":matthews_corrcoef(yva,pred),
        })

per_fold=pd.DataFrame(rows)
per_fold.to_csv(OUTDIR/"exp13_k1_vs_k5_per_fold.csv",index=False)

summary=(
    per_fold.groupby("kernel")
    .agg({
        "accuracy":["mean","std"],"precision":["mean","std"],
        "recall":["mean","std"],"specificity":["mean","std"],
        "f1":["mean","std"],"roc_auc":["mean","std"],
        "pr_auc":["mean","std"],"balanced_accuracy":["mean","std"],
        "mcc":["mean","std"],
    })
)
display(summary)
summary.to_csv(OUTDIR/"exp13_k1_vs_k5_summary.csv")


## 10. Interpretation

This experiment isolates the local-neighborhood size much more cleanly than the thesis pilot kernel experiment because:
- the same 577-patient MGMT cohort is used;
- the same patient-level target is used;
- the same 24 GLCM features are used;
- the same `%A` representation is used;
- the same 10 folds and XGBoost model are used;
- only `kernelRadius=1` vs `kernelRadius=5` changes.

The metrics remain MGMT patient-level metrics and must be reported separately from the thesis Tumor/Non-tumor voxel-level metrics.
